# DCGAN on CIFAR-10

这个 Notebook 完整展示 `DCGAN（Deep Convolutional GAN）` 在 `CIFAR-10` 上训练图像生成的流程。

内容包括：
- CIFAR-10 数据加载与预处理
- Generator 与 Discriminator 实现
- GAN 对抗训练机制解读（博弈论视角）
- G/D 交替训练流程
- 训练过程中生成样本可视化
- 与 DDPM 扩散模型的对比分析

## 1. 环境准备

```bash
pip install torch torchvision matplotlib
```

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.utils as vutils

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    data_root: str = './data'
    image_size: int = 32
    batch_size: int = 128
    num_workers: int = 2
    # 潜在向量维度，Generator 从噪声空间映射到图像空间
    nz: int = 100
    # Generator 特征图基准通道数
    ngf: int = 64
    # Discriminator 特征图基准通道数
    ndf: int = 64
    nc: int = 3
    # 较小学习率防止 G/D 训练失衡
    lr: float = 2e-4
    beta1: float = 0.5
    epochs: int = 30
    # 每隔多少步保存一次生成图像
    log_interval: int = 100

cfg = Config()
cfg

## 2. 加载 CIFAR-10

In [ ]:
# 归一化到 [-1, 1]，与 Generator 末层 tanh 输出范围一致
transform = transforms.Compose([
    transforms.Resize(cfg.image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = datasets.CIFAR10(root=cfg.data_root, train=True, download=True, transform=transform)
print(f'训练集大小：{len(train_dataset)}')

## 3. 构建 DataLoader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

# 展示真实图像样本
real_batch = next(iter(train_loader))
fig, ax = plt.subplots(figsize=(10, 4))
grid = vutils.make_grid(real_batch[0][:32] * 0.5 + 0.5, nrow=8, padding=2)
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.set_title('真实 CIFAR-10 样本')
ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Generator 实现

Generator 接受随机噪声向量 `z`（形状 `[B, nz, 1, 1]`），通过一系列 `ConvTranspose2d` 逐步上采样到 `[B, 3, 32, 32]`。

- **BatchNorm**：稳定训练，防止梯度消失
- **ReLU**：线性区间大，梯度流动好
- **Tanh**：输出归一化到 [-1, 1]，与真实图像归一化范围一致

In [ ]:
class Generator(nn.Module):
    def __init__(self, nz, ngf, nc):
        super().__init__()
        self.net = nn.Sequential(
            # nz -> ngf*4, 1x1 -> 4x4
            nn.ConvTranspose2d(nz, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # 4x4 -> 8x8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # 8x8 -> 16x16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # 16x16 -> 32x32
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


netG = Generator(cfg.nz, cfg.ngf, cfg.nc).to(device)

# 正态初始化权重，GAN 对初始化较敏感
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)

netG.apply(weights_init)
print(netG)

## 5. Discriminator 实现

Discriminator 把图像 `[B, 3, 32, 32]` 压缩为一个标量概率，判断输入是真实图像还是生成图像。

- **LeakyReLU**：负区间有小梯度，防止梯度死亡
- **不用 MaxPool**：让 D 自己学习降采样方式

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, nc, ndf):
        super().__init__()
        self.net = nn.Sequential(
            # 32x32 -> 16x16
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 16x16 -> 8x8
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # 8x8 -> 4x4
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # 4x4 -> 1x1
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).view(-1)


netD = Discriminator(cfg.nc, cfg.ndf).to(device)
netD.apply(weights_init)
print(netD)

## 6. GAN 训练机制解读

### 6.1 博弈论视角

GAN 是一个 **二人零和博弈**：
- **G（Generator）**：目标是生成逼真图像，骗过 D
- **D（Discriminator）**：目标是区分真图和假图

### 6.2 损失函数

$$\min_G \max_D \; \mathbb{E}[\log D(x)] + \mathbb{E}[\log(1 - D(G(z)))]$$

- **D 的目标**：最大化 `log D(real) + log(1 - D(fake))`
- **G 的目标**：最小化 `log(1 - D(G(z)))`，实践中改为最大化 `log D(G(z))` 避免梯度消失

### 6.3 与 DDPM 对比

| 维度 | DCGAN | DDPM |
|------|-------|------|
| 训练方式 | 对抗博弈 | 最大化似然（去噪目标） |
| 生成速度 | 一步前向传播，极快 | 需要数百步迭代去噪 |
| 训练稳定性 | 容易不稳定（mode collapse） | 训练稳定 |
| 图像质量 | 中等 | 更高（当前 SOTA） |

## 7. 参数量统计

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Generator 参数量    : {count_parameters(netG):,}')
print(f'Discriminator 参数量: {count_parameters(netD):,}')

## 8. 训练函数（G/D 交替）

In [ ]:
criterion = nn.BCELoss()

# 两个优化器独立更新，防止 G 更新干扰 D 的梯度
optimizerD = optim.Adam(netD.parameters(), lr=cfg.lr, betas=(cfg.beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=cfg.lr, betas=(cfg.beta1, 0.999))

# 固定的测试噪声，用于对比不同 epoch 的生成质量
fixed_noise = torch.randn(64, cfg.nz, 1, 1, device=device)


def train_one_epoch(netG, netD, dataloader, optimizerG, optimizerD, criterion, device, nz):
    netG.train()
    netD.train()
    d_losses, g_losses = [], []

    for real_images, _ in dataloader:
        real_images = real_images.to(device)
        b = real_images.size(0)

        # --- 训练 Discriminator ---
        netD.zero_grad()
        # label smoothing：真实标签用 0.9 而非 1，防止 D 过于自信
        real_labels = torch.full((b,), 0.9, device=device)
        fake_labels = torch.zeros(b, device=device)

        d_real = netD(real_images)
        loss_d_real = criterion(d_real, real_labels)

        noise = torch.randn(b, nz, 1, 1, device=device)
        fake_images = netG(noise)
        d_fake = netD(fake_images.detach())
        loss_d_fake = criterion(d_fake, fake_labels)

        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        optimizerD.step()

        # --- 训练 Generator ---
        netG.zero_grad()
        # G 希望 D 把假图判为真，所以目标标签是 1
        d_fake_for_g = netD(fake_images)
        loss_g = criterion(d_fake_for_g, torch.ones(b, device=device))
        loss_g.backward()
        optimizerG.step()

        d_losses.append(loss_d.item())
        g_losses.append(loss_g.item())

    return sum(d_losses) / len(d_losses), sum(g_losses) / len(g_losses)

## 9. 训练主循环

In [ ]:
history = {'d_loss': [], 'g_loss': []}
image_snapshots = []

for epoch in range(cfg.epochs):
    d_loss, g_loss = train_one_epoch(
        netG, netD, train_loader, optimizerG, optimizerD, criterion, device, cfg.nz
    )
    history['d_loss'].append(d_loss)
    history['g_loss'].append(g_loss)

    # 每 5 个 epoch 保存一次生成结果快照
    if (epoch + 1) % 5 == 0 or epoch == 0:
        with torch.no_grad():
            fake = netG(fixed_noise).detach().cpu()
        image_snapshots.append((epoch + 1, fake))

    print(f'Epoch {epoch+1:3d}/{cfg.epochs}  D_loss={d_loss:.4f}  G_loss={g_loss:.4f}')

In [ ]:
# G/D loss 应大致平衡；若 D_loss 趋近于 0，说明 G 训练失败（mode collapse 信号）
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['d_loss'], label='Discriminator loss')
ax.plot(history['g_loss'], label='Generator loss')
ax.set_title('GAN 训练损失曲线')
ax.set_xlabel('Epoch')
ax.legend()
plt.tight_layout()
plt.show()

## 10. 生成图像展示与分析

In [ ]:
fig, axes = plt.subplots(1, len(image_snapshots), figsize=(4 * len(image_snapshots), 4))
if len(image_snapshots) == 1:
    axes = [axes]

for ax, (ep, imgs) in zip(axes, image_snapshots):
    grid = vutils.make_grid(imgs[:16] * 0.5 + 0.5, nrow=4, padding=2)
    ax.imshow(grid.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(f'Epoch {ep}')
    ax.axis('off')

plt.suptitle('DCGAN 不同训练阶段生成效果', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 潜在空间插值：在两个噪声向量之间线性插值，观察生成图像的连续变化
z1 = torch.randn(1, cfg.nz, 1, 1, device=device)
z2 = torch.randn(1, cfg.nz, 1, 1, device=device)
alphas = torch.linspace(0, 1, 8, device=device)

with torch.no_grad():
    interp_imgs = torch.cat([
        netG((1 - a) * z1 + a * z2) for a in alphas
    ])

grid = vutils.make_grid(interp_imgs * 0.5 + 0.5, nrow=8, padding=2)
fig, ax = plt.subplots(figsize=(14, 3))
ax.imshow(grid.cpu().permute(1, 2, 0).clamp(0, 1).numpy())
ax.set_title('潜在空间插值：z1 → z2')
ax.axis('off')
plt.tight_layout()
plt.show()